# Demostración del Orquestador Multi-Agente Jerárquico (Pre-entrega 6)

Este notebook demuestra el flujo interactivo de delegación y supervisión implementado con **LangGraph**, **LangChain** y **Pydantic**.

### Componentes del Sistema:
1. **Supervisor (`agents/supervisor.py`)**: Router inteligente y controlador de flujo que decide dinámicamente qué especialista debe actuar.
2. **ResearchAgent (`agents/research_agent.py`)**: Especialista de búsqueda y telemetría de clústeres.
3. **AnalystAgent (`agents/analyst_agent.py`)**: Especialista de cálculo numérico de SLA y estrés de recursos.
4. **Validador Determinístico (`agents/validator.py`)**: Puerta de calidad en código Python puro (*'Pydantic, no vibras'*).


## 1. Importación de Módulos y Verificación del Grafo

In [ ]:
import os
import asyncio
from dotenv import load_dotenv

from state import MultiAgentState, ResearchArtifact, AnalysisArtifact, ValidationArtifact
from tools import query_cluster_metrics, calculate_availability_score, calculate_resource_stress_index
from agents.validator import deterministic_validator_node
from graph import build_multi_agent_graph

# Cargar variables de entorno si existen
load_dotenv()

graph = build_multi_agent_graph()
print("Grafo Multi-Agente compilado exitosamente.")
print(f"Nodos registrados: {list(graph.nodes.keys())}")

## 2. Visualización de la Topología Jerárquica
El flujo sigue el patrón Supervisor con aristas de retorno hacia el router.

In [ ]:
from IPython.display import Image, display

try:
    png_data = graph.get_graph().draw_mermaid_png()
    display(Image(png_data))
except Exception as e:
    print("Mermaid en formato texto:")
    print(graph.get_graph().draw_mermaid())

## 3. Demostración de las Herramientas Especializadas
Cada especialista cuenta con herramientas funcionales acotadas con validación estricta de esquemas Pydantic.

In [ ]:
# Herramienta del ResearchAgent: Telemetría de infraestructura
telemetria = query_cluster_metrics.invoke({"cluster_id": "cloud-cluster-prod"})
print("=== Telemetría Recolectada por el Researcher ===")
print(telemetria)

# Herramientas del AnalystAgent: Cálculos de SLA y Estrés de Hardware
disp = calculate_availability_score.invoke({"total_requests": 850000, "error_requests": 4250})
estres = calculate_resource_stress_index.invoke({"cpu_pct": 88.4, "memory_pct": 91.2})

print("\n=== Cálculos Numéricos del Analyst ===")
print(f"Disponibilidad: {disp['availability_percentage']}% (SLA Cumplido: {disp['sla_met']})")
print(f"Índice de Estrés: {estres['stress_index']}/100 - Categoría: {estres['risk_category']}")

## 4. Demostración de la Puerta de Calidad (Validación Determinística)
El nodo validador inspecciona la consistencia cruzada entre lo investigado y lo calculado, sin depender de un LLM.

In [ ]:
# Caso 1: Datos y cálculos consistentes (Aprobado)
research_ok = ResearchArtifact(
    topic="Auditoría de nodos",
    fuentes_consultadas=["telemetry_db:prod"],
    datos_extraidos={
        "total_requests": 850000,
        "error_requests": 4250,
        "cpu_utilization_pct": 88.4,
        "memory_utilization_pct": 91.2
    },
    confidence_score=0.95
)

analysis_ok = AnalysisArtifact(
    availability_rate=99.50,
    stress_index=90.08,
    risk_category="CRITICO",
    key_findings=["Saturación de memoria con OOM killer activo"],
    confidence_score=0.98
)

estado_simulado = {
    "researcher_output": research_ok,
    "analyst_output": analysis_ok,
    "messages": []
}

res_val_ok = deterministic_validator_node(estado_simulado)
val_art_ok = res_val_ok["validation_output"]
print("=== Caso Coherente ===")
print(f"Válido: {val_art_ok.valido}")
print(f"Conflictos: {val_art_ok.conflictos_detectados}")
print(f"Escalación Humana: {val_art_ok.requiere_escalacion_humana}")

# Caso 2: Contradicción detectada (Hardware saturado pero catalogado como BAJO riesgo)
analysis_incoherente = AnalysisArtifact(
    availability_rate=99.50,
    stress_index=90.08,
    risk_category="BAJO",  # ¡Contradicción!
    key_findings=["Todo en orden"],
    confidence_score=0.98
)

estado_conflicto = {
    "researcher_output": research_ok,
    "analyst_output": analysis_incoherente,
    "messages": []
}

res_val_err = deterministic_validator_node(estado_conflicto)
val_art_err = res_val_err["validation_output"]
print("\n=== Caso con Conflicto / Contradicción ===")
print(f"Válido: {val_art_err.valido}")
print(f"Conflictos: {val_art_err.conflictos_detectados}")
print(f"Escalación Humana Requerida: {val_art_err.requiere_escalacion_humana}")

## 5. Inspección de la Traza de Ejecución Multi-Paso
Visualizamos la traza completa guardada en `multi_agent_trace.log`.

In [ ]:
with open('multi_agent_trace.log', 'r', encoding='utf-8') as f:
    print(f.read())